In [ ]:
# -----------------------------导入python库------------------------------------
import os
import netCDF4 as nc
import pandas as pd
import numpy as np
import numpy.ma as ma
import xarray as xr
import matplotlib as mpl
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.style as mplstyle
import cartopy.crs as ccrs
import netCDF4 as nc
from netCDF4 import Dataset
import shapefile
import time
from datetime import datetime, timedelta

import cmaps
BASE_PATH = "/home/yfdong/data/work/SMP"
DATA_PATH = "/raid61/yfdong/data/work/SMP"
import sys
# 将目标模块所在的路径添加到 sys.path 中
sys.path.append(os.path.join(BASE_PATH, "code", "Library"))
from MeteoVarPlot import draw_2DVAR, draw_Bias, draw_VARtemporal, draw_Bias_subplot,draw_2DVAR_subplot ,draw_2DuvVAR ,draw_uvBias_subplot ,draw_2DuvVAR_subplot
from MeteoChartPlot import add_TickGrid
from PreprocessVar import (
    get_MONTH_abbr, get_pentad, get_letter,
    preprocess_var, read_var,Events_configs,
    Read_VarsFromVarsDict, experiments_types,return_Event_Date,
    load_geodata, SubsetDomainConfigure ,create_domain_mask,get_letter,
    
)
Era5Land_VarResults={}
Wrfout_VarResults={}
Obs_DataPath = os.path.join(DATA_PATH, "Evaluation/Obs")
Sim_DataPath = os.path.join(DATA_PATH, "output/dst/TotalVar")
# Era5Land_VarList = ['sp','u10','v10','t2m','q2m','tp','e','swvl1']
# Wrfout_VarList = ['PSFC', 'U10', 'V10', 'T2', 'Q2', 'RAINNC', 'ET', 'SH2O_Layer1']
# -----------------------------读取数据------------------------------------
# LSM_VarList = ['T2m', 'Q2m', 'Prec', 'SM', 'ET', 'SH', 'PSFC'] #'Wind10m', 'T2m', 'PSFC', 'ET', 'Q2m', 'Wind10m', 'Prec'
LSM_VarList = ['SM', 'SH', 'LH', 'RUNSF', 'FIRA'] # 'Wind10m', 'T2m', 'PSFC', 'ET', 'Q2m', 'Wind10m', 'Prec', 'SPEI''ET', 'ST', 
# LSM_VarList = ['ET']
# 读取地形数据
# LAT, LON, LandMask, DEM = load_geodata(GeoPath =os.path.join(DATA_PATH, "Domain/geo_em.d01.nc") ,IF_LandMask=True)
LAT, LON, LandMask, DEM = load_geodata(GeoPath ="/home/yfdong/data/work/SMP/data/Domain/geo_em.d01.nc" ,IF_LandMask=True)
# Timedelta = 24  # 选取的wrfout时间间隔（hour）
CalibrationStatus = 'def'
LAYER = 0
START_YEAR = 2011
END_YEAR = 2011
# YearNum = END_YEAR - START_YEAR + 1
# Years = [2016, 2020]
Years = list(range(START_YEAR, END_YEAR + 1))
YearNum = len(Years)
StartReadMonth, EndReadMonth = 6, 8
StartDay, EndDay = 1, 31
CtrlName, ExpName = 'Ctrl', 'FSM'
# 字典
sim_var_ScaleFactor = {
    'ET': 1,
    'SH': 1,
    'LH': 1,
    'Q2m': 1, # 
    'SM': 100,
    'ST': 1,
    'Wind10m': 1,
    'T2m': 1,
    'T2max': 1,
    'T2min': 1,
    'Prec': 24, # mm/h -> mm/d
    'PSFC': 1, 
    'SPEI': 1,
    'FIRA': 1,
    'RUNSF': 1,
}
sim_var_mapping = {
    'SM': 'SH2O_Layer1',  # 土壤湿度第一层
    'ST': 'TSLB_Layer1',
    'ET': 'ET',
    'SH': 'SH',
    'LH': 'LH',
    'Q2m': 'Q2',
    'T2m': 'T2',
    'T2max': 'T2max',
    'T2min': 'T2min',
    'PSFC': 'PSFC',
    'Wind10m': 'Wind10',
    'Prec': 'RAINNC',
    'SPEI': 'SPEI_1d',
    'FIRA': 'FIRA',
    'RUNSF': 'RUNSF',
}
VarUnit = {
    'SM': '10⁻² m³/m³',
    'ST': 'K',
    'ET': 'mm/day',
    'LH': 'W/m²',
    'SH': 'W/m²',
    'FIRA': 'W/m²',
    'Q2m': 'g/kg',
    'T2m': 'K',
    'T2max': 'K',
    'T2min': 'K',
    'PSFC': 'hPa',
    'Wind10m': 'm/s',
    'Prec': 'mm/day',
    'RUNSF': 'mm/day',
    'SPEI': '-',
}
ExpStates = [CtrlName, ExpName]


In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm
ResultDict = {
    CtrlName: {},
    ExpName: {},
}
def _read_one(args):
    ExpState, Year, Var = args
    """读取单个 (ExpState, Year, Var) 的数据"""
    start_str, end_str = f'{Year}-05-06 00:00:00', f'{Year}-08-30 00:00:00'
    fmt = '%Y-%m-%d %H:%M:%S'
    Read_StartDate = datetime.strptime(start_str, fmt)
    Read_EndDate   = datetime.strptime(end_str,   fmt)

    SimVarName = sim_var_mapping[Var]
    SimFilePath = os.path.join(
        Sim_DataPath, str(Year),
        f"{Year}-def-{ExpState}-{SimVarName}_1-Hourly_MEAN.nc"
    )

    with xr.open_dataset(SimFilePath) as ds:
        data = (ds[SimVarName]
                .sel(time=slice(Read_StartDate, Read_EndDate))
                .values * sim_var_ScaleFactor[Var])
    return ExpState, Year, Var, data

if __name__ == "__main__":
    tasks = []
    for ExpState in ExpStates:
        for Year in Years:
            ResultDict[ExpState][Year] = {}      # ← 关键：先建好空 dict
            for Var in LSM_VarList:
                tasks.append((ExpState, Year, Var))

    with ProcessPoolExecutor(max_workers=30) as ex:
        for fut in tqdm(as_completed([ex.submit(_read_one, t) for t in tasks]),
                        total=len(tasks)):
            ExpState, Year, Var, data = fut.result()
            ResultDict[ExpState][Year][Var] = data


In [ ]:
ResultDict[CtrlName][Year][Var].shape

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def extract_SubsetRegion_mask(var, domain_mask):
    row_indices, col_indices = np.where(domain_mask == 1)
    row_start, row_end = row_indices.min(), row_indices.max()
    col_start, col_end = col_indices.min(), col_indices.max()
    if len(var.shape) == 2:
        sub_var = var[row_start:row_end+1, col_start:col_end+1]
    elif len(var.shape) == 3:
        sub_var = var[:, row_start:row_end+1, col_start:col_end+1]
    elif len(var.shape) == 4:
        sub_var = var[:, :, row_start:row_end+1, col_start:col_end+1]
    else:
        raise ValueError("输入数据的维度不正确，请输入二维或三维数组。")
    return sub_var

# ============================================================
# 0. 参数设置
# ============================================================
SubsetRegions = {
    'HHP': [35, 40, 114.5, 119],
    'LPT': [35, 40, 108, 114.5],
    'QDW': [28.5, 35, 108, 112.5],
    'YHP': [28.5, 35, 112.5, 122],
    'SEH': [23, 28.5, 110, 120]
}
RegionNames = list(SubsetRegions.keys())
N = len(RegionNames)

plt_VarList = ['SM', 'SH', 'LH', 'RUNSF', 'FIRA']
# plt_VarList = ['SM']
for Var in plt_VarList:
    # ============================================================
    # 1. 逐区域处理：切片 → 时间平均
    # ============================================================
    RegionData = {}
    for reg in RegionNames:
        mask = create_domain_mask(LAT, LON, SubsetRegions[reg])
        FSM_sim  = extract_SubsetRegion_mask(ResultDict[ExpName][Year][Var], mask)
        Ctrl_sim = extract_SubsetRegion_mask(ResultDict[CtrlName][Year][Var], mask)

        FSM_1d  = np.nanmean(FSM_sim,  axis=(1, 2))
        Ctrl_1d = np.nanmean(Ctrl_sim, axis=(1, 2))
        RegionData[reg] = (FSM_1d, Ctrl_1d)

    # ============================================================
    # 2. 构造逐小时时间轴
    # ============================================================
    nt = len(RegionData[RegionNames[0]][0])
    time_hourly = pd.date_range(
        start=f'{START_YEAR}-05-06 00:00',
        end=f'{START_YEAR}-08-30 00:00',
        freq='h'
    )[:nt]

    RegionData_hr = {}
    for reg in RegionNames:
        FSM_1d, Ctrl_1d = RegionData[reg]
        FSM_s  = pd.Series(FSM_1d,  index=time_hourly)
        Ctrl_s = pd.Series(Ctrl_1d, index=time_hourly)
        RegionData_hr[reg] = (FSM_s, Ctrl_s)

    # ============================================================
    # 3. 时间参数 & 绿色虚线时刻（只在可见范围内）
    # ============================================================
    x_start = pd.Timestamp(f'{START_YEAR}-06-15 00:00')
    x_end   = pd.Timestamp(f'{START_YEAR}-07-15 00:00')
    line_start = pd.Timestamp(f'{START_YEAR}-05-05 08:00')
    line_times = pd.date_range(start=line_start, end=x_end, freq='48h')
    line_times_vis = [t for t in line_times if x_start <= t <= x_end]

    # ============================================================
    # 4. 统计每个区域：
    #    (a) 净冲击：Mean(|ΔFSM - ΔCTRL|)  （扣除自然变化）
    #    (b) CTRL 的平均值
    # ============================================================
    RegionStats = {}
    for reg in RegionNames:
        FSM_s, Ctrl_s = RegionData_hr[reg]
        Ctrl_vis = Ctrl_s.loc[x_start:x_end].values

        dlist_net = []
        for t in line_times_vis:
            idx = FSM_s.index.get_indexer([t], method='nearest')[0]
            if 0 < idx < len(FSM_s):
                # 计算净冲击：FSM差分 - CTRL差分
                diff_FSM = FSM_s.iloc[idx+1] - FSM_s.iloc[idx]
                diff_Ctrl = Ctrl_s.iloc[idx+1] - Ctrl_s.iloc[idx]
                dlist_net.append(abs(diff_FSM - diff_Ctrl))

        net_impact = np.nanmean(dlist_net) if len(dlist_net) else np.nan
        ctrl_mean  = np.nanmean(Ctrl_vis)

        RegionStats[reg] = dict(
            net_impact=net_impact,
            ctrl_mean=ctrl_mean
        )

    # ============================================================
    # 5. 绘图：N×2，左列时间序列，右列柱状图
    # ============================================================
    fig, axes = plt.subplots(
        nrows=N, ncols=2,
        figsize=(11.*0.6, 2.0 * N * 0.6),
        dpi=350,
        gridspec_kw={'width_ratios': [6, 1]}
    )
    if N == 1:
        axes = np.array([axes])

    # Nature 风格配色
    color_net = 'GREY'  # 深蓝
    color_ctrl = '#73A1C2' # 橙黄

    for i, reg in enumerate(RegionNames):
        # ---------------- 左列：时间序列 ----------------
        ax = axes[i, 0]
        FSM_s, Ctrl_s = RegionData_hr[reg]

        ax.plot(FSM_s.index,  FSM_s.values,  color='#FF8C69', lw=1.3, label=f'{ExpName}')
        ax.plot(Ctrl_s.index, Ctrl_s.values, color='#73A1C2', lw=1.3, linestyle='--', label='CTRL')

        for t in line_times_vis:
            ax.axvline(t, color='green', ls='--', lw=0.8, alpha=0.4)

        ax.set_xlim(x_start, x_end)

        y_data = np.concatenate([
            FSM_s.loc[x_start:x_end].values,
            Ctrl_s.loc[x_start:x_end].values
        ])
        y_min, y_max = np.nanmin(y_data), np.nanmax(y_data)
        pad = (y_max - y_min) * 0.05 if y_max > y_min else 0.5
        ax.set_ylim(y_min - pad, y_max + pad)

        panel_label = f"({chr(ord('a') + i)}) {reg}"
        ax.text(0.01, 0.92, panel_label, transform=ax.transAxes,
                fontsize=10, fontweight='bold', va='top', ha='left')

        ax.grid(True, axis='y', ls='--', alpha=0.4)
        ax.tick_params(axis='y', labelsize=8)
        if i < N - 1:
            ax.tick_params(axis='x', labelbottom=False)
        # ---------------- 右列：柱状图 ----------------
        axb = axes[i, 1]
        st = RegionStats[reg]
        x = np.arange(2)   # 0=Net Impact, 1=CTRL Mean
        w = 0.5
        vals = [st['net_impact'], st['ctrl_mean']]
        colors = [color_net, color_ctrl]

        b = axb.bar(x, vals, w, color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)

        axb.set_xticks(x)
        if i == N - 1:
            axb.set_xticklabels(['NI', 'CM'], fontsize=8)
        else:
            axb.set_xticklabels([])  # 隐藏 X 轴标签，避免重叠

        axb.tick_params(axis='y', labelsize=7)
        axb.grid(True, axis='y', ls='--', alpha=0.4)

        # 柱顶数值标注（根据高度动态调整位置）
        for rect, val in zip(b, vals):
            h = rect.get_height()
            if np.isfinite(h):
                offset = 2 if h >= 0 else -10
                va = 'bottom' if h >= 0 else 'top'
                axb.annotate(f'{h:.2f}',
                             xy=(rect.get_x() + rect.get_width()/2, h),
                             xytext=(0, offset), textcoords='offset points',
                             ha='center', va=va, fontsize=6)

    # ============================================================
    # 6. 坐标轴美化与图例
    # ============================================================
    axes[-1, 0].xaxis.set_major_locator(mdates.DayLocator(interval=5))
    axes[-1, 0].xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    plt.setp(axes[-1, 0].get_xticklabels(), rotation=0, ha='center', fontsize=8)

    # 时间序列图例
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels,
               loc='lower center',
               bbox_to_anchor=(0.4, 0.02),
               ncol=2, frameon=False, fontsize=9)

    fig.text(0.04, 0.5, f"{Var} ({VarUnit[Var]})",
             va='center', ha='left', rotation='vertical', fontsize=11)

    plt.subplots_adjust(hspace=0.2, wspace=0.15)
    plt.show()